In [7]:
import pandas as pd
import numpy as np

In [8]:
#load processed weather dataset
weather_df = pd.read_csv(r"C:\AgroGuard AI\data\processed\weather_data_processed.csv")


In [19]:
weather_df.head()

,date,YEAR,DOY,state,temp_avg,temp_max,temp_min,rainfall,humidity,solar_radiation,wind_speed,heat_stress,rainfall_7day_avg,consecutive_dry_days,consecutive_hot_days,temp_range,month,risk_label
0,2015-01-01,2015,1,Benue,25.88,34.28,18.35,0.0,39.40,21.11,2.30,0,0.0,1,0,15.93,1,1
1,2015-01-02,2015,2,Benue,24.06,33.75,15.99,0.0,39.66,22.30,3.08,0,0.0,2,0,17.76,1,1
2,2015-01-03,2015,3,Benue,23.49,32.06,16.87,0.0,35.63,17.92,4.04,0,0.0,3,0,15.19,1,1
3,2015-01-04,2015,4,Benue,23.06,31.34,17.29,0.0,29.73,20.46,4.39,0,0.0,4,0,14.05,1,0
4,2015-01-05,2015,5,Benue,21.94,29.39,16.53,0.0,29.82,18.19,4.13,0,0.0,5,0,12.86,1,0


In [13]:
weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10959 entries, 0 to 10958
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  10959 non-null  str    
 1   YEAR                  10959 non-null  int64  
 2   DOY                   10959 non-null  int64  
 3   state                 10959 non-null  str    
 4   temp_avg              10959 non-null  float64
 5   temp_max              10959 non-null  float64
 6   temp_min              10959 non-null  float64
 7   rainfall              10959 non-null  float64
 8   humidity              10959 non-null  float64
 9   solar_radiation       10959 non-null  float64
 10  wind_speed            10959 non-null  float64
 11  heat_stress           10959 non-null  int64  
 12  rainfall_7day_avg     10959 non-null  float64
 13  consecutive_dry_days  10959 non-null  int64  
 14  consecutive_hot_days  10959 non-null  int64  
 15  temp_range            10959 no

In [14]:
weather_df['date'] = pd.to_datetime(weather_df['date'])

In [15]:
#define crop parameters
# Base temperature and maturity threshold for each crop
CROP_PARAMS = {
    'maize': {
        'base_temp': 10,
        'maturity_gdd': 1350,  # midpoint of 1200-1500 range
        'description': 'Maize/Corn'
    },
    'cassava': {
        'base_temp': 8,
        'maturity_gdd': 2000,
        'description': 'Cassava'
    },
    'yam': {
        'base_temp': 12,
        'maturity_gdd': 1800,
        'description': 'Yam'
    }
}

In [16]:
#define a function to calculate GDD
def calculate_daily_gdd(temp_max, temp_min, base_temp):
    """
    Calculate Growing Degree Days for a single day.
    Uses the average temperature method.
    """
    avg_temp = (temp_max + temp_min) / 2
    gdd = max(0, avg_temp - base_temp)  
    # max(0) ensures we never get negative GDD
    return round(gdd, 2)

In [17]:
def calculate_days_to_maturity(weather_df, planting_date, state, crop_type):
    """
    Calculate how many days until crop reaches maturity
    from the planting date.
    
    Returns:
    - days_to_maturity: number of days
    - maturity_date: actual calendar date
    - accumulated_gdd: total GDD at maturity
    - daily_gdd_log: day by day GDD record
    """
    
    #get crop parameters
    crop = CROP_PARAMS[crop_type]
    base_temp = crop['base_temp']
    maturity_threshold = crop['maturity_gdd']
    
    #filter weather data from planting date for the right state
    planting_date = pd.to_datetime(planting_date)
    farmer_weather = weather_df[
        (weather_df['date'] >= planting_date) & 
        (weather_df['state'] == state)
    ].copy().reset_index(drop=True)
    
    if len(farmer_weather) == 0:
        return None, None, None, None
    
    #accumulate GDD day by day
    accumulated_gdd = 0
    daily_gdd_log = []
    maturity_day = None
    maturity_date = None
    
    for idx, row in farmer_weather.iterrows():
        daily_gdd = calculate_daily_gdd(
            row['temp_max'], 
            row['temp_min'], 
            base_temp
        )
        accumulated_gdd += daily_gdd
        daily_gdd_log.append({
            'day': idx + 1,
            'date': row['date'],
            'temp_max': row['temp_max'],
            'temp_min': row['temp_min'],
            'daily_gdd': daily_gdd,
            'accumulated_gdd': round(accumulated_gdd, 2)
        })
        
        #check if maturity threshold reached
        if accumulated_gdd >= maturity_threshold and maturity_day is None:
            maturity_day = idx + 1
            maturity_date = row['date']
            break  #stop once maturity is reached
    
    return maturity_day, maturity_date, round(accumulated_gdd, 2), daily_gdd_log

In [20]:
#test with a sample 
planting_date = "2022-04-01"
state = "Kaduna"
crop_type = "maize"

days, maturity_date, total_gdd, gdd_log = calculate_days_to_maturity(weather_df, 
                                                                     planting_date, state, crop_type)

print(f"Crop: {CROP_PARAMS[crop_type]['description']}")
print(f"State: {state}")
print(f"Planting Date: {planting_date}")
print(f"Days to Maturity: {days} days")
print(f"Expected Maturity Date: {maturity_date.strftime('%Y-%m-%d') if maturity_date else 'N/A'}")
print(f"Total GDD Accumulated: {total_gdd}")

#show first 5 days of GDD log
gdd_df = pd.DataFrame(gdd_log)
print("\nDaily GDD Log (first 5 days):")
print(gdd_df.head())
print("\nDaily GDD Log (last 5 days):")
print(gdd_df.tail())

Crop: Maize/Corn
State: Kaduna
Planting Date: 2022-04-01
Days to Maturity: 91 days
Expected Maturity Date: 2022-06-30
Total GDD Accumulated: 1354.94

Daily GDD Log (first 5 days):
   day       date  temp_max  temp_min  daily_gdd  accumulated_gdd
0    1 2022-04-01     35.33     18.69      17.01            17.01
1    2 2022-04-02     33.31     19.90      16.61            33.62
2    3 2022-04-03     29.45     20.27      14.86            48.48
3    4 2022-04-04     31.45     19.41      15.43            63.91
4    5 2022-04-05     30.50     21.71      16.11            80.02

Daily GDD Log (last 5 days):
    day       date  temp_max  temp_min  daily_gdd  accumulated_gdd
86   87 2022-06-26     27.00     21.27      14.13          1299.90
87   88 2022-06-27     27.31     20.58      13.95          1313.85
88   89 2022-06-28     26.59     21.11      13.85          1327.70
89   90 2022-06-29     27.24     20.76      14.00          1341.70
90   91 2022-06-30     25.97     20.51      13.24          

In [21]:
test_scenarios = [
    ('maize', 'Kaduna', '2022-04-01'),
    ('cassava', 'Benue', '2022-03-15'),
    ('yam', 'Ogun', '2022-04-10')
]

print("=" * 60)
print("GDD MATURITY PREDICTIONS - ALL CROPS")
print("=" * 60)

for crop, state, plant_date in test_scenarios:
    days, mat_date, total_gdd, _ = calculate_days_to_maturity(
        weather_df, plant_date, state, crop
    )
    
    if days:
        print(f"\nCrop: {CROP_PARAMS[crop]['description']}")
        print(f"State: {state} | Planted: {plant_date}")
        print(f"Matures in: {days} days ({mat_date.strftime('%B %d, %Y')})")
        print(f"Total GDD: {total_gdd}")
    else:
        print(f"\nCould not calculate for {crop} in {state}")

print("\n" + "=" * 60)

GDD MATURITY PREDICTIONS - ALL CROPS

Crop: Maize/Corn
State: Kaduna | Planted: 2022-04-01
Matures in: 91 days (June 30, 2022)
Total GDD: 1354.94

Crop: Cassava
State: Benue | Planted: 2022-03-15
Matures in: 106 days (June 28, 2022)
Total GDD: 2015.87

Crop: Yam
State: Ogun | Planted: 2022-04-10
Matures in: 130 days (August 17, 2022)
Total GDD: 1801.05



In [22]:
#save the processed gdd log for maize as a sample
gdd_df.to_csv('../data/processed/gdd_maize_sample.csv', index=False)
print("GDD log saved successfully!")

GDD log saved successfully!
